<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/MiniMax_H3_Director_FL2V_Turbo_Colab_A100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniMax H3 导演台｜FL2V Turbo｜Google Colab Pro A100

本版本面向 **t2v / i2v / fl2v**，保留原导演台三段首尾帧时间线，并加入官方 LightX2V：

- LoRA：`minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors`
- 默认：**8 steps**（Cell 1 可切换为 4 steps）
- Sigma shift：`12 / 3`
- SageAttention 与 Refine 默认旁路，优先保证 A100 稳定运行

> 不要用于 r2v / v2v / rv2v；这些任务请使用同包内的 Ref2V Turbo 版本。

### v2 高速下载

启用 Hugging Face Xet 高性能模式：默认 64 路 Range GET，并行下载 4 个模型文件；中断后重跑 Cell 4 会复用缓存并断点续传。

### v3 访问修复

ComfyUI v19+ 会拒绝非 localhost 域名的请求并返回 HTTP 403；Cell 6 已自动加上 `--enable-cors-header`，并默认通过 Cloudflare 临时道打开界面。


In [ ]:
#@title Cell 1｜配置与 A100 环境检查
import gc
import json
import os
import re
import shlex
import shutil
import urllib.request
import subprocess
import sys
import time
from pathlib import Path

# -------------------- 可调参数 --------------------
MODEL_VARIANT = "auto" #@param ["auto", "int8_convrot", "fp8_scaled"]
TURBO_STEPS = 8 #@param [8, 4]
PARALLEL_FILE_DOWNLOADS = 4 #@param [2, 3, 4, 5]
XET_RANGE_WORKERS = 64 #@param [16, 32, 64]
ACCESS_MODE = "cloudflare" #@param ["cloudflare", "colab_proxy", "both"]
ENABLE_CORS_HEADER = True #@param {type:"boolean"}
SAVE_OUTPUTS_TO_DRIVE = False #@param {type:"boolean"}
FAST_FP16_ACCUMULATION = True #@param {type:"boolean"}
PORT = 8188 #@param {type:"integer"}

ROOT = Path("/content")
WORK_ROOT = ROOT / "minimax_h3_director_fl2v_turbo_a100"
COMFY = ROOT / "ComfyUI"
HF_CACHE = ROOT / "hf_cache"
WF_ORIGINAL = WORK_ROOT / "workflow_original.json"
WF_ACTIVE = WORK_ROOT / "MiniMax_H3_Director_FL2V_Turbo_A100.json"
LOG_FILE = WORK_ROOT / "comfyui.log"
PID_FILE = WORK_ROOT / "comfyui.pid"
MODEL_REPO = "Comfy-Org/MiniMax-H3"
LORA_REPO = "lightx2v/Minimax-h3-Turbo"
LORA_FILE = "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors"
TASK_FAMILY = "fl2va"
WORKFLOW_NAME = "MiniMax_H3_Director_FL2V_Turbo_A100"

for p in (WORK_ROOT, HF_CACHE):
    p.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"] = str(XET_RANGE_WORKERS)
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
os.environ["HF_XET_RECONSTRUCT_WRITE_SEQUENTIALLY"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


def run(cmd, cwd=None, check=True, capture=False, env=None):
    """运行命令；列表参数避免 shell 转义问题。"""
    if isinstance(cmd, str):
        shown = cmd
        shell = True
    else:
        shown = shlex.join(map(str, cmd))
        shell = False
    print("$", shown)
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        shell=shell,
        check=False,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
        env=env,
    )
    if check and result.returncode:
        if capture and result.stdout:
            print(result.stdout[-5000:])
        raise RuntimeError(f"命令失败（exit={result.returncode}）：{shown}")
    return result.stdout if capture else result.returncode


def free_gb(path=ROOT):
    return shutil.disk_usage(path).free / 1024**3


def ram_gb():
    with open("/proc/meminfo", encoding="utf-8") as f:
        kb = int(re.search(r"MemTotal:\s+(\d+)", f.read()).group(1))
    return kb / 1024**2


def torch_probe():
    probe = (
        "import json,torch; "
        "print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,"
        "'available':torch.cuda.is_available(),"
        "'name':torch.cuda.get_device_name(0) if torch.cuda.is_available() else '',"
        "'cap':list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else []}))"
    )
    out = run([sys.executable, "-c", probe], capture=True, check=False)
    try:
        return json.loads(out.strip().splitlines()[-1])
    except Exception:
        return {'torch': '', 'cuda': '', 'available': False, 'name': '', 'cap': []}


# Colab Secrets 中可选配置 HF_TOKEN
try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN")
    if _token:
        os.environ["HF_TOKEN"] = _token
        print("✓ 已从 Colab Secrets 读取 HF_TOKEN")
except Exception:
    pass

# 用 nvidia-smi 检查硬件，不在 Notebook 内核中 import torch 占用显存
smi = run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
    capture=True,
    check=False,
).strip()
if not smi:
    raise RuntimeError("未检测到 NVIDIA GPU：运行时 → 更改运行时类型 → 选择 A100 GPU")
print("GPU:", smi.splitlines()[0])
if "A100" not in smi.upper():
    print("⚠ 当前不是 A100；本笔记本仍可继续，但参数只针对 A100 验证。")

PROBE = torch_probe()
if not PROBE.get("available"):
    raise RuntimeError("PyTorch 看不到 GPU，请重新连接 GPU 运行时。")
print(f"PyTorch {PROBE['torch']} | CUDA {PROBE['cuda']} | {PROBE['name']} | cap={PROBE['cap']}")
print(f"系统内存 {ram_gb():.1f} GB | 可用磁盘 {free_gb():.1f} GB")
if ram_gb() < 40:
    print("⚠ 建议在 Colab Pro 中启用高 RAM；长时间线生成可能耗尽系统内存。")
if free_gb() < 60:
    raise RuntimeError("可用磁盘不足 60 GB。请使用干净的 Colab 运行时或清理 /content。")

cuda_major = int((PROBE.get("cuda") or "0").split(".")[0] or 0)
if MODEL_VARIANT == "auto":
    EFFECTIVE_VARIANT = "int8_convrot" if cuda_major >= 13 else "fp8_scaled"
else:
    EFFECTIVE_VARIANT = MODEL_VARIANT
if EFFECTIVE_VARIANT == "int8_convrot" and cuda_major < 13:
    print("⚠ 你强制选择了 INT8 ConvRot，但当前 PyTorch CUDA < 13；若加载失败请改回 auto。")
print("DiT 方案:", EFFECTIVE_VARIANT)

if SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

CONFIG = {
    "task_family": TASK_FAMILY,
    "turbo_steps": int(TURBO_STEPS),
    "parallel_file_downloads": int(PARALLEL_FILE_DOWNLOADS),
    "xet_range_workers": int(XET_RANGE_WORKERS),
    "access_mode": ACCESS_MODE,
    "lora_file": LORA_FILE,
    "model_variant": MODEL_VARIANT,
    "effective_variant": EFFECTIVE_VARIANT,
    "save_outputs_to_drive": SAVE_OUTPUTS_TO_DRIVE,
    "fast_fp16_accumulation": FAST_FP16_ACCUMULATION,
    "port": PORT,
    "torch": PROBE,
}
(WORK_ROOT / "config.json").write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")
print("✓ 配置完成")

# ---- 访问层工具：ComfyUI v19+ 对非 localhost 的 Host 会返回 403，必须开启 CORS ----
def start_cloudflare_tunnel(port, timeout=90):
    """启动 cloudflared 临时道，返回公网地址。"""
    binary = ROOT / "cloudflared"
    tunnel_log = ROOT / "cloudflared.log"
    subprocess.run(["pkill", "-x", "cloudflared"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if not binary.exists():
        print("下载 cloudflared ...")
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            str(binary),
        )
        binary.chmod(0o755)
    handle = tunnel_log.open("w", encoding="utf-8")
    proc = subprocess.Popen(
        [str(binary), "tunnel", "--url", f"http://127.0.0.1:{port}", "--protocol", "http2", "--no-autoupdate"],
        stdout=handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    handle.close()
    for _ in range(timeout):
        time.sleep(1)
        text = tunnel_log.read_text(encoding="utf-8", errors="ignore") if tunnel_log.exists() else ""
        found = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
        if found:
            return found.group(0), proc
        if proc.poll() is not None:
            break
    print(tunnel_log.read_text(encoding="utf-8", errors="ignore")[-2000:])
    return None, proc


def remote_host_allowed(port, fake_host="probe.trycloudflare.com"):
    """模拟道请求，确认 ComfyUI 不会回 403。"""
    request = urllib.request.Request(f"http://127.0.0.1:{port}/", headers={"Host": fake_host})
    try:
        with urllib.request.urlopen(request, timeout=10) as response:
            return response.status, None
    except urllib.error.HTTPError as exc:
        return exc.code, None
    except Exception as exc:
        return None, repr(exc)


In [ ]:
#@title Cell 2｜释放内嵌 FL2V Turbo 工作流并对齐模型/LoRA
import base64
import hashlib

WORKFLOW_B64 = """..."""
EXPECTED_SHA256 = "9bceb8eab776d5a635674fdf14bbdb3977abc2ebabf239506f97587878ccb297"

raw = base64.b64decode(WORKFLOW_B64)
actual_sha = hashlib.sha256(raw).hexdigest()
assert actual_sha == EXPECTED_SHA256, "内嵌工作流校验失败"
WF_ORIGINAL.write_bytes(raw)
workflow = json.loads(raw.decode("utf-8"))

DIT_FILE = (
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors"
    if EFFECTIVE_VARIANT == "int8_convrot"
    else "minimax_h3_fl2va_pruned_fp8_scaled.safetensors"
)
TEXT_FILE = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
VIDEO_VAE_FILE = "minimax_h3_video_vae_fp16.safetensors"
AUDIO_VAE_FILE = "minimax_h3_audio_vae_fp32.safetensors"


def nodes_of_type(wf, node_type):
    return [n for n in wf.get("nodes", []) if n.get("type") == node_type]


def set_loader_model(node, filename, directory=None):
    values = node.setdefault("widgets_values", [])
    if not values:
        values.append(filename)
    else:
        values[0] = filename
    for model in node.get("properties", {}).get("models", []) or []:
        model["name"] = filename
        if directory:
            model["directory"] = directory


for node in nodes_of_type(workflow, "UNETLoader"):
    set_loader_model(node, DIT_FILE, "diffusion_models")
for node in nodes_of_type(workflow, "CLIPLoader"):
    set_loader_model(node, TEXT_FILE, "text_encoders")
    if len(node.get("widgets_values", [])) > 1:
        node["widgets_values"][1] = "minimax"
for node in nodes_of_type(workflow, "VAELoader"):
    title = (node.get("title") or "").lower()
    set_loader_model(node, AUDIO_VAE_FILE if "audio" in title else VIDEO_VAE_FILE, "vae")
for node in nodes_of_type(workflow, "LoraLoaderModelOnly"):
    set_loader_model(node, LORA_FILE, "loras")
    if len(node.get("widgets_values", [])) < 2:
        node.setdefault("widgets_values", []).append(1.0)
    else:
        node["widgets_values"][1] = 1.0

# A100 稳定优先：可选 Sage 与 Refine/放大保持旁路。
for node in workflow.get("nodes", []):
    if node.get("type") in {
        "PathchSageAttentionKJ",
        "MiniMaxH3MemoryEfficientSageAttentionPatch",
        "MiniMaxH3DirectorRefine",
        "UpscaleModelLoader",
    }:
        node["mode"] = 4


def director_node(wf):
    found = nodes_of_type(wf, "MiniMaxH3Director")
    return found[0] if found else None


def timeline_from_director(node):
    if not node:
        return None, -1
    for i, value in enumerate(node.get("widgets_values", [])):
        if not isinstance(value, str) or not value.lstrip().startswith("{"):
            continue
        try:
            obj = json.loads(value)
        except Exception:
            continue
        if isinstance(obj, dict) and "segments" in obj:
            return obj, i
    return None, -1


def ordered_assets(timeline):
    seen, result = set(), []
    def add(name):
        if isinstance(name, str) and name and name not in seen:
            seen.add(name)
            result.append(name)
    def walk(obj):
        if isinstance(obj, dict):
            add(obj.get("imageFile"))
            for value in obj.values():
                walk(value)
        elif isinstance(obj, list):
            for value in obj:
                walk(value)
    walk(timeline or {})
    return result


director = director_node(workflow)
if director is None:
    raise RuntimeError("未找到 MiniMaxH3Director 节点")
if len(director.get("widgets_values", [])) <= 17:
    raise RuntimeError("导演台节点版本不匹配：高级采样控件缺失")
director["widgets_values"][13] = int(TURBO_STEPS)
director["widgets_values"][14] = "res_multistep"
director["widgets_values"][15] = "simple"
director["widgets_values"][16] = 12
director["widgets_values"][17] = 3

timeline, TIMELINE_WIDGET_INDEX = timeline_from_director(director)
ASSETS = ordered_assets(timeline) if TASK_FAMILY == "fl2va" else []
WF_ACTIVE.write_text(json.dumps(workflow, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")

print("✓ 工作流释放完成")
print("  工作流      :", WORKFLOW_NAME)
print("  任务族      :", TASK_FAMILY)
print("  DiT         :", DIT_FILE)
print("  Turbo LoRA  :", LORA_FILE)
print("  采样        :", TURBO_STEPS, "steps / res_multistep / simple")
print("  Sigma shift :", 12, "/", 3)
print("  任务        :", director.get("widgets_values", ["?"])[0])
print("  首尾帧素材  :", len(ASSETS))


In [ ]:
#@title Cell 3｜安装/更新 ComfyUI 与 FL2V Turbo 所需节点
import importlib.metadata as importlib_metadata


def sync_repo(url, path, branch=None):
    path = Path(path)
    if (path / ".git").exists():
        print("更新", path.name)
        run(["git", "fetch", "--depth", "1", "origin"], cwd=path)
        target = f"origin/{branch}" if branch else "FETCH_HEAD"
        if branch:
            run(["git", "checkout", "-q", branch], cwd=path, check=False)
        run(["git", "reset", "--hard", "-q", target], cwd=path)
    else:
        print("安装", path.name)
        cmd = ["git", "clone", "--depth", "1"]
        if branch:
            cmd += ["--branch", branch]
        cmd += [url, str(path)]
        run(cmd)


def pip_install(args):
    run([sys.executable, "-m", "pip", "install", "-q"] + list(args))


def install_requirements_safely(req_path):
    """保留 Colab 自带 torch/triton；可选 Sage 节点在 JSON 中本来就是旁路。"""
    req_path = Path(req_path)
    if not req_path.exists():
        return
    blocked = re.compile(r"^\s*(torch|torchvision|torchaudio|triton|sageattention)(?:\b|[<>=!~])", re.I)
    lines = []
    for line in req_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#") or blocked.match(stripped):
            continue
        lines.append(line)
    if lines:
        filtered = WORK_ROOT / (req_path.parent.name + "_requirements.txt")
        filtered.write_text("\n".join(lines) + "\n", encoding="utf-8")
        pip_install(["-r", str(filtered)])


sync_repo("https://github.com/comfyanonymous/ComfyUI.git", COMFY, branch="master")
try:
    torch_before = importlib_metadata.version("torch")
except Exception:
    torch_before = ""

pip_install(["-r", str(COMFY / "requirements.txt")])
pip_install(["--upgrade", "huggingface_hub", "hf_xet", "requests", "pillow"])

try:
    torch_after = importlib_metadata.version("torch")
except Exception:
    torch_after = ""
if torch_before and torch_after != torch_before:
    print(f"⚠ torch 版本发生变化：{torch_before} → {torch_after}。如 GPU 探测失败，请重启运行时后重跑。")

custom_nodes = COMFY / "custom_nodes"
custom_nodes.mkdir(parents=True, exist_ok=True)
PLUGIN_REPOS = {
    "ComfyUI_MiniMaxH3_Director": "https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git",
    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes.git",
}
for name, url in PLUGIN_REPOS.items():
    target = custom_nodes / name
    sync_repo(url, target)
    install_requirements_safely(target / "requirements.txt")

# 将改写后的工作流放进 ComfyUI 左侧 Workflows 列表
workflow_dir = COMFY / "user" / "default" / "workflows"
workflow_dir.mkdir(parents=True, exist_ok=True)
INSTALLED_WORKFLOW = workflow_dir / f"{WORKFLOW_NAME}.json"
shutil.copy2(WF_ACTIVE, INSTALLED_WORKFLOW)

if SAVE_OUTPUTS_TO_DRIVE:
    drive_output = Path("/content/drive/MyDrive/MiniMax_H3_Director/output")
    drive_output.mkdir(parents=True, exist_ok=True)
    local_output = COMFY / "output"
    if local_output.is_symlink():
        local_output.unlink()
    elif local_output.exists():
        for item in local_output.iterdir():
            dst = drive_output / item.name
            if not dst.exists():
                shutil.move(str(item), str(dst))
        shutil.rmtree(local_output)
    os.symlink(drive_output, local_output)
    print("✓ 输出目录已指向", drive_output)

commit = run(["git", "log", "-1", "--format=%h %ad %s", "--date=short"], cwd=COMFY, capture=True).strip()
print("✓ ComfyUI:", commit)
print("✓ 自定义节点: MiniMaxH3 Director + KJNodes")
print("✓ 工作流已安装:", INSTALLED_WORKFLOW)


In [ ]:
#@title Cell 4｜HF Xet 高带宽并行下载模型与 Turbo LoRA
# Colab 高带宽优化：
# 1) hf_xet 对单个大文件并发 Range GET；2) ThreadPoolExecutor 同时下载多个文件。
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import HfApi, hf_hub_download

PARALLEL_FILE_DOWNLOADS = int(globals().get("PARALLEL_FILE_DOWNLOADS", 4))
XET_RANGE_WORKERS = int(globals().get("XET_RANGE_WORKERS", 64))
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"] = str(XET_RANGE_WORKERS)
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
os.environ["HF_XET_RECONSTRUCT_WRITE_SEQUENTIALLY"] = "0"  # Colab NVMe 适合并行写
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"

MODEL_SPECS = [
    (MODEL_REPO, f"diffusion_models/{DIT_FILE}", "diffusion_models"),
    (MODEL_REPO, f"text_encoders/{TEXT_FILE}", "text_encoders"),
    (MODEL_REPO, f"vae/{VIDEO_VAE_FILE}", "vae"),
    (MODEL_REPO, f"vae/{AUDIO_VAE_FILE}", "vae"),
    (LORA_REPO, LORA_FILE, "loras"),
]
TOKEN = os.environ.get("HF_TOKEN") or None

print(f"HF Xet 高性能模式：开启 | 单文件 Range 并发：{XET_RANGE_WORKERS} | 文件并行：{PARALLEL_FILE_DOWNLOADS}")
print("提示：Xet 会分别显示 downloading bytes 与 reconstructing file；后者是本地重建速度。")


def valid_safetensors(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size < 1024:
        return False
    if path.suffix != ".safetensors":
        return True
    with path.open("rb") as f:
        header_len = int.from_bytes(f.read(8), "little")
        first = f.read(1)
    return 1 < header_len < 256 * 1024 * 1024 and first == b"{"


def link_or_copy(src, dst):
    src, dst = Path(os.path.realpath(src)), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if valid_safetensors(dst) and dst.stat().st_size == src.stat().st_size:
        return "已存在"
    tmp = dst.with_name(dst.name + ".installing")
    if tmp.exists() or tmp.is_symlink():
        tmp.unlink()
    try:
        os.link(src, tmp)
        mode = "硬链接"
    except OSError:
        shutil.copy2(src, tmp)
        mode = "复制"
    os.replace(tmp, dst)
    if not valid_safetensors(dst):
        raise RuntimeError(f"文件校验失败：{dst}")
    return mode


def destination(spec):
    repo_id, remote_name, subdir = spec
    return COMFY / "models" / subdir / Path(remote_name).name


# 读取文件大小，仅用于磁盘预检；失败不影响下载。
sizes = {}
for repo_id in sorted({spec[0] for spec in MODEL_SPECS}):
    try:
        info = HfApi().model_info(repo_id, files_metadata=True, token=TOKEN)
        for item in info.siblings:
            sizes[(repo_id, item.rfilename)] = item.size or 0
    except Exception as exc:
        print(f"⚠ 无法读取 {repo_id} 文件大小，将继续下载：", repr(exc))

pending = []
missing_bytes = 0
for spec in MODEL_SPECS:
    repo_id, remote_name, _ = spec
    dst = destination(spec)
    if valid_safetensors(dst):
        print(f"✓ 已存在：{dst.name} ({dst.stat().st_size / 1024**3:.2f} GB)")
    else:
        pending.append(spec)
        missing_bytes += sizes.get((repo_id, remote_name), 0)

if missing_bytes:
    need_gb = missing_bytes / 1024**3
    print(f"预计新增下载 {need_gb:.1f} GB；当前可用 {free_gb():.1f} GB")
    if need_gb and free_gb() < need_gb + 8:
        raise RuntimeError("磁盘不足：至少还需下载量 + 8 GB 余量。")


def download_one(spec):
    repo_id, remote_name, _ = spec
    dst = destination(spec)
    # hf_hub_download 会自动复用并续传 /content/hf_cache 中的 Xet/LFS 分块。
    cached = hf_hub_download(
        repo_id=repo_id,
        filename=remote_name,
        cache_dir=str(HF_CACHE),
        token=TOKEN,
    )
    mode = link_or_copy(cached, dst)
    return dst, mode


if pending:
    workers = max(1, min(PARALLEL_FILE_DOWNLOADS, len(pending)))
    print(f"\n开始并行下载 {len(pending)} 个文件，文件级线程数：{workers}")
    failures = []
    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="hf-download") as pool:
        futures = {pool.submit(download_one, spec): spec for spec in pending}
        for future in as_completed(futures):
            repo_id, remote_name, _ = futures[future]
            try:
                dst, mode = future.result()
                print(f"✓ 完成：{dst.name} | {mode} | {dst.stat().st_size / 1024**3:.2f} GB")
            except Exception as exc:
                failures.append((repo_id, remote_name, exc))
                print(f"✗ 失败：{repo_id}/{remote_name} | {exc!r}")
    if failures:
        details = "\n".join(f"- {repo}/{name}: {exc!r}" for repo, name, exc in failures)
        raise RuntimeError("部分文件下载失败；重新运行本格会断点续传：\n" + details)
else:
    print("\n✓ 所有文件均已存在，无需下载。")

# 完成后逐个校验，避免某个线程静默留下不完整文件。
missing_or_bad = [str(destination(spec)) for spec in MODEL_SPECS if not valid_safetensors(destination(spec))]
if missing_or_bad:
    raise RuntimeError("下载后校验失败：\n" + "\n".join(missing_or_bad))

shutil.copy2(WF_ACTIVE, INSTALLED_WORKFLOW)
print("\n✓ 模型与 Turbo LoRA 全部就绪；剩余磁盘 %.1f GB" % free_gb())
print("✓ 当前 DiT :", DIT_FILE)
print("✓ 当前 LoRA:", LORA_FILE)


In [ ]:
#@title Cell 5｜依次上传 4 张首尾帧（也可跳过，进 ComfyUI 后再换图）
UPLOAD_ASSETS = True #@param {type:"boolean"}
REPLACE_EXISTING = False #@param {type:"boolean"}

from io import BytesIO
from PIL import Image, ImageOps

input_dir = COMFY / "input"
input_dir.mkdir(parents=True, exist_ok=True)
roles = [
    "片段 1 首帧",
    "片段 1 尾帧 / 片段 2 首帧",
    "片段 2 尾帧 / 片段 3 首帧",
    "片段 3 尾帧",
]


def update_dimensions(obj, filename, width, height):
    if isinstance(obj, dict):
        if obj.get("imageFile") == filename:
            obj["width"], obj["height"] = width, height
        for value in obj.values():
            update_dimensions(value, filename, width, height)
    elif isinstance(obj, list):
        for value in obj:
            update_dimensions(value, filename, width, height)


if not UPLOAD_ASSETS:
    print("已跳过。打开 ComfyUI 后可在导演台时间线里重新选择图片。")
else:
    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError("此单元格需要在 Google Colab 中运行") from exc

    for i, target_name in enumerate(ASSETS):
        target = input_dir / target_name
        role = roles[i] if i < len(roles) else f"素材 {i + 1}"
        if target.exists() and not REPLACE_EXISTING:
            print(f"✓ {role}: 已存在 {target.name}")
            continue
        print(f"\n[{i + 1}/{len(ASSETS)}] 请选择：{role}")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError(f"没有收到 {role}，请重新运行本格。")
        source_name, payload = next(iter(uploaded.items()))
        with Image.open(BytesIO(payload)) as im:
            im = ImageOps.exif_transpose(im)
            if im.mode not in ("RGB", "RGBA"):
                im = im.convert("RGB")
            width, height = im.size
            # 工作流引用名以 .png 结尾，因此统一真实转码为 PNG，而不是只改后缀。
            im.save(target, format="PNG", optimize=False)
        update_dimensions(timeline, target_name, width, height)
        print(f"✓ {source_name} → {target.name} ({width}×{height})")

    if director and timeline is not None and TIMELINE_WIDGET_INDEX >= 0:
        director["widgets_values"][TIMELINE_WIDGET_INDEX] = json.dumps(
            timeline, ensure_ascii=False, separators=(",", ":")
        )
    WF_ACTIVE.write_text(json.dumps(workflow, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
    shutil.copy2(WF_ACTIVE, INSTALLED_WORKFLOW)

missing = [name for name in ASSETS if not (input_dir / name).exists()]
if missing:
    print("⚠ 仍缺素材；可在 ComfyUI 导演台中重新选择：")
    for name in missing:
        print("  -", name)
else:
    print("\n✓ 4 张首尾帧全部就绪，工作流已回写真实尺寸。")


In [ ]:
#@title Cell 6｜启动 ComfyUI（修复 403）、体检节点并生成访问地址
import re
import signal
import urllib.error
import urllib.request

ACCESS_MODE = globals().get("ACCESS_MODE", "cloudflare")
ENABLE_CORS_HEADER = bool(globals().get("ENABLE_CORS_HEADER", True))

# 停止本笔记本上一次启动的进程
if PID_FILE.exists():
    try:
        old_pid = int(PID_FILE.read_text().strip())
        os.kill(old_pid, signal.SIGTERM)
        time.sleep(2)
    except Exception:
        pass
subprocess.run(["pkill", "-f", "main.py --listen"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

args = [
    sys.executable,
    "main.py",
    "--listen", "127.0.0.1",
    "--port", str(PORT),
    "--preview-method", "auto",
    "--disable-auto-launch",
]
help_text = run([sys.executable, "main.py", "--help"], cwd=COMFY, capture=True, check=False) or ""
if ENABLE_CORS_HEADER:
    if "--enable-cors-header" in help_text:
        # ComfyUI v19+ 对代理/道域名的 Host 会返回 HTTP 403，开启后才能远程打开。
        args += ["--enable-cors-header"]
    else:
        print("⚠ 当前 ComfyUI 不支持 --enable-cors-header，已忽略。")
if FAST_FP16_ACCUMULATION:
    if "--fast" in help_text:
        args += ["--fast", "fp16_accumulation"]
    else:
        print("⚠ 当前 ComfyUI 不支持 --fast，已自动忽略。")

log_handle = open(LOG_FILE, "w", encoding="utf-8")
process = subprocess.Popen(
    args,
    cwd=COMFY,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
PID_FILE.write_text(str(process.pid), encoding="utf-8")
print("启动命令:", shlex.join(args))

base_url = f"http://127.0.0.1:{PORT}"
ready = False
for _ in range(180):
    if process.poll() is not None:
        break
    try:
        with urllib.request.urlopen(base_url + "/system_stats", timeout=2) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    log_handle.flush()
    tail = LOG_FILE.read_text(encoding="utf-8", errors="ignore")[-8000:]
    print(tail)
    raise RuntimeError("ComfyUI 未能启动；上方已打印日志尾部。")

with urllib.request.urlopen(base_url + "/object_info", timeout=30) as response:
    object_info = json.loads(response.read().decode("utf-8"))

required_nodes = [
    "UNETLoader", "CLIPLoader", "VAELoader",
    "MiniMaxH3Director", "CreateVideo", "SaveVideo", "PreviewAny",
    "LoraLoaderModelOnly",
]
optional_bypassed_nodes = [
    "PathchSageAttentionKJ",
    "MiniMaxH3MemoryEfficientSageAttentionPatch",
    "MiniMaxH3DirectorRefine",
    "UpscaleModelLoader",
]
missing_required = [name for name in required_nodes if name not in object_info]
missing_optional = [name for name in optional_bypassed_nodes if name not in object_info]

model_paths = [
    COMFY / "models" / "diffusion_models" / DIT_FILE,
    COMFY / "models" / "text_encoders" / TEXT_FILE,
    COMFY / "models" / "vae" / VIDEO_VAE_FILE,
    COMFY / "models" / "vae" / AUDIO_VAE_FILE,
    COMFY / "models" / "loras" / LORA_FILE,
]
missing_models = [str(p) for p in model_paths if not valid_safetensors(p)]
missing_assets = [name for name in ASSETS if not (COMFY / "input" / name).exists()]

remote_status, remote_error = remote_host_allowed(PORT)
if remote_error:
    remote_text = "未能探测：" + remote_error
elif remote_status == 403:
    remote_text = "403（远程域名会被拒绝，请确认已开启 CORS）"
else:
    remote_text = f"通过（HTTP {remote_status}）"

print("\n" + "=" * 72)
print("ComfyUI       : READY")
print("工作流        : Workflows →", WORKFLOW_NAME)
print("任务族        :", TASK_FAMILY)
print("DiT           :", DIT_FILE)
print("Turbo LoRA    :", LORA_FILE)
print("采样步数      :", TURBO_STEPS)
print("主链节点      :", "全部就绪" if not missing_required else "缺少 " + ", ".join(missing_required))
print("旁路节点      :", "全部可识别" if not missing_optional else "缺少 " + ", ".join(missing_optional))
print("模型          :", "全部就绪" if not missing_models else f"缺 {len(missing_models)} 个")
print("首尾帧        :", "全部就绪" if not missing_assets else f"缺 {len(missing_assets)} 个，可进导演台重选")
print("远程 Host 校验 :", remote_text)
print("=" * 72)

if missing_required:
    raise RuntimeError("主链缺少节点，请重跑 Cell 3，再重跑 Cell 6。")
if missing_models:
    raise RuntimeError("模型未就绪，请重跑 Cell 4。")

links = []
if ACCESS_MODE in ("cloudflare", "both"):
    tunnel_url, tunnel_proc = start_cloudflare_tunnel(PORT)
    globals()["_cloudflared_process"] = tunnel_proc
    if tunnel_url:
        links.append(("Cloudflare 道（推荐）", tunnel_url))
    else:
        print("⚠ Cloudflare 道未能建立，请重跑本格或改用 Colab 代理。")
if ACCESS_MODE in ("colab_proxy", "both") or not links:
    try:
        from google.colab.output import eval_js
        links.append(("Colab 代理", eval_js(f"google.colab.kernel.proxyPort({PORT})")))
    except Exception as exc:
        print("⚠ 无法取得 Colab 代理地址：", repr(exc))

print()
for label, url in links:
    print(f"{label}: {url}")
try:
    from IPython.display import HTML, display
    html = "<br>".join(
        f'<a href="{url}" target="_blank" style="font-size:18px;font-weight:700">在新标签页打开 ComfyUI（{label}）</a>'
        for label, url in links
    )
    display(HTML(html))
except Exception:
    pass
print("日志文件：", LOG_FILE)
print("cloudflared 日志：", ROOT / "cloudflared.log")


In [ ]:
#@title Cell 7｜下载最新成片（生成完成后运行）
DOWNLOAD_ALL_OUTPUTS = False #@param {type:"boolean"}

output_dir = COMFY / "output"
video_exts = {".mp4", ".mov", ".webm", ".mkv"}
videos = sorted(
    [p for p in output_dir.rglob("*") if p.is_file() and p.suffix.lower() in video_exts],
    key=lambda p: p.stat().st_mtime,
)
if not videos:
    raise FileNotFoundError("还没有找到成片，请先在 ComfyUI 中运行工作流。")

from google.colab import files
if DOWNLOAD_ALL_OUTPUTS:
    archive_base = str(WORK_ROOT / "MiniMax_H3_outputs")
    archive = shutil.make_archive(archive_base, "zip", output_dir)
    print("下载全部输出：", archive)
    files.download(archive)
else:
    latest = videos[-1]
    print("下载最新成片：", latest)
    files.download(str(latest))


## 使用与排错

1. **执行顺序**：Cell 1 → 6；生成完成后运行 Cell 7 下载成片。
2. **用途**：本版本只用于 `t2v / i2v / fl2v`，默认保留原来的三段首尾帧时间线；不要切换到 `r2v / v2v / rv2v`。
3. **采样**：默认使用 FL2VA Turbo 8-step v1.0；Cell 1 可把 `TURBO_STEPS` 改为 4 进行更快预览。
4. **首尾帧**：Cell 5 会依次要求上传原时间线需要的 4 张图片，也可跳过后在导演台重新选择。
5. **完整质量**：在 ComfyUI 中旁路 LoRA，再把 steps 改为 20–25；Turbo 与完整模式不会生成完全相同的镜头。
6. **INT8 加载失败**：Cell 1 将 `MODEL_VARIANT` 改为 `fp8_scaled`，再重跑 Cell 2、4、5、6。
7. **CUDA OOM**：先把每段 124 帧降至 85 帧，或把输出降到约 0.3MP；建议 A100 + 高 RAM。
8. **Sage/Refine 灰色**：为稳定性默认旁路；确认主流程可用后再尝试开启。
9. **Colab 会话断开**：如需保留成片，在 Cell 1 勾选 `SAVE_OUTPUTS_TO_DRIVE`。

> 权重受各自许可约束；本笔记本不会绕过 Hugging Face 的访问控制。

11. **下载速度**：Cell 1 可调整 `PARALLEL_FILE_DOWNLOADS` 与 `XET_RANGE_WORKERS`；A100 Colab 默认 4 / 64。中断后直接重跑 Cell 4 会断点续传。

12. **打开界面报 403**：ComfyUI v19+ 会校验 Host；Cell 1 保持 `ENABLE_CORS_HEADER = True`，`ACCESS_MODE` 选 `cloudflare`，重跑 Cell 6 即可。临时道地址为公开链接，请勿外传。
